# AI-Powered Employee Attrition Prediction & Workforce Risk Intelligence System

## Machine Learning Model Development & Evaluation

### Objective
The objective of this notebook is to develop and evaluate machine learning models capable of predicting employee attrition risk.

### Key Goals
- Prepare training and testing datasets
- Handle class imbalance
- Train multiple machine learning models
- Evaluate model performance
- Compare predictive accuracy
- Select the best-performing model for deployment

In [1]:
# ============================================
# Import Required Libraries
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

import joblib

# Display settings
pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")

Libraries imported successfully.


# Load Final Processed Dataset

The machine learning-ready dataset is loaded for predictive model development.

In [3]:
# ============================================
# Load Final Dataset
# ============================================

df = pd.read_csv("../data/final_employee_attrition.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [4]:
# Display dataset preview
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,IncomeExperienceRatio,PromotionDelay,EngagementScore,WorkStressFlag
0,41,1,2,1102,2,1,2,1,2,0,94,3,2,7,4,2,5993,19479,8,1,11,3,1,0,8,0,1,6,4,0,5,665.888889,0,8,0
1,49,0,1,279,1,8,1,1,3,1,61,2,2,6,2,1,5130,24907,1,0,23,4,4,1,10,3,3,10,7,1,7,466.363636,0,12,0
2,37,1,2,1373,1,2,2,4,4,1,92,2,1,2,3,2,2090,2396,6,1,15,3,2,0,7,3,3,0,0,0,0,261.250000,0,12,0
3,33,0,1,1392,1,3,4,1,4,0,56,3,1,6,3,1,2909,23159,1,1,11,3,3,0,8,3,3,8,7,3,0,323.222222,0,13,1
4,27,0,2,591,1,2,1,3,1,1,40,3,1,2,2,1,3468,16632,9,0,12,3,4,1,6,3,3,2,2,2,2,495.428571,0,10,0


# Feature and Target Separation

The dataset is divided into:
- Features (X)
- Target variable (y)

### Target Variable
- Attrition
    - 1 → Employee left
    - 0 → Employee stayed

In [5]:
# ============================================
# Separate Features and Target Variable
# ============================================

X = df.drop('Attrition', axis=1)

y = df['Attrition']

print("Feature-target separation completed.")

Feature-target separation completed.


# Train-Test Split

The dataset is divided into:
- Training set (80%)
- Testing set (20%)

Stratification is applied to preserve class distribution.

In [6]:
# ============================================
# Train-Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training Set Shape :", X_train.shape)
print("Testing Set Shape  :", X_test.shape)

Training Set Shape : (1176, 34)
Testing Set Shape  : (294, 34)


# Handling Class Imbalance Using SMOTE

Employee attrition datasets are often imbalanced.

SMOTE (Synthetic Minority Oversampling Technique) is applied to:
- Improve minority class representation
- Reduce model bias
- Improve attrition prediction capability

In [7]:
# ============================================
# Apply SMOTE
# ============================================

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train,
    y_train
)

print("SMOTE applied successfully.")

SMOTE applied successfully.


# Feature Scaling

Feature scaling standardizes numerical feature values to improve:
- Model convergence
- Training stability
- Prediction performance

StandardScaler is used for normalization.

In [8]:
# ============================================
# Apply Feature Scaling
# ============================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_resampled)

X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed successfully.")

Feature scaling completed successfully.


# Logistic Regression Model

Logistic Regression serves as a baseline classification model.

Advantages:
- Simple and interpretable
- Fast training
- Useful benchmark for comparison

In [9]:
# ============================================
# Train Logistic Regression Model
# ============================================

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_scaled, y_train_resampled)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [10]:
# Generate predictions

log_predictions = log_model.predict(X_test_scaled)

log_probabilities = log_model.predict_proba(X_test_scaled)[:, 1]

# Model Evaluation Function

A reusable evaluation function is created to measure:
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC Score

In [11]:
# ============================================
# Model Evaluation Function
# ============================================

def evaluate_model(y_true, predictions, probabilities):

    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions)
    recall = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)
    roc_auc = roc_auc_score(y_true, probabilities)

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(roc_auc, 4))

    print("\nClassification Report:\n")
    print(classification_report(y_true, predictions))

In [12]:
# Evaluate Logistic Regression Model

evaluate_model(
    y_test,
    log_predictions,
    log_probabilities
)

Accuracy : 0.8367
Precision: 0.4884
Recall   : 0.4468
F1 Score : 0.4667
ROC-AUC  : 0.7506

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.91      0.90       247
           1       0.49      0.45      0.47        47

    accuracy                           0.84       294
   macro avg       0.69      0.68      0.69       294
weighted avg       0.83      0.84      0.83       294



# Random Forest Classifier

Random Forest is an ensemble learning algorithm capable of:
- Handling non-linear relationships
- Capturing feature interactions
- Reducing overfitting through ensemble averaging

In [13]:
# ============================================
# Train Random Forest Model
# ============================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train_resampled, y_train_resampled)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [14]:
# Generate predictions

rf_predictions = rf_model.predict(X_test)

rf_probabilities = rf_model.predict_proba(X_test)[:, 1]

In [15]:
# Evaluate Random Forest Model

evaluate_model(
    y_test,
    rf_predictions,
    rf_probabilities
)

Accuracy : 0.8061
Precision: 0.375
Recall   : 0.3191
F1 Score : 0.3448
ROC-AUC  : 0.7401

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.90      0.89       247
           1       0.38      0.32      0.34        47

    accuracy                           0.81       294
   macro avg       0.62      0.61      0.62       294
weighted avg       0.79      0.81      0.80       294



# XGBoost Classifier

XGBoost is a high-performance gradient boosting algorithm widely used for:
- Structured/tabular data
- Classification tasks
- High predictive accuracy
- Handling feature complexity

In [16]:
# ============================================
# Train XGBoost Model
# ============================================

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_resampled, y_train_resampled)

print("XGBoost model trained successfully.")

XGBoost model trained successfully.


In [17]:
# Generate predictions

xgb_predictions = xgb_model.predict(X_test)

xgb_probabilities = xgb_model.predict_proba(X_test)[:, 1]

In [18]:
# Evaluate XGBoost Model

evaluate_model(
    y_test,
    xgb_predictions,
    xgb_probabilities
)

Accuracy : 0.8299
Precision: 0.4634
Recall   : 0.4043
F1 Score : 0.4318
ROC-AUC  : 0.7419

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.91      0.90       247
           1       0.46      0.40      0.43        47

    accuracy                           0.83       294
   macro avg       0.68      0.66      0.67       294
weighted avg       0.82      0.83      0.83       294



# Model Performance Comparison

This section compares the performance of all trained machine learning models to identify the best-performing solution for deployment.

In [19]:
# ============================================
# Model Comparison Table
# ============================================

model_results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'XGBoost'
    ],
    'Accuracy': [
        accuracy_score(y_test, log_predictions),
        accuracy_score(y_test, rf_predictions),
        accuracy_score(y_test, xgb_predictions)
    ],
    'F1 Score': [
        f1_score(y_test, log_predictions),
        f1_score(y_test, rf_predictions),
        f1_score(y_test, xgb_predictions)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, log_probabilities),
        roc_auc_score(y_test, rf_probabilities),
        roc_auc_score(y_test, xgb_probabilities)
    ]
})

model_results

,Model,Accuracy,F1 Score,ROC-AUC
0,Logistic Regression,0.836735,0.466667,0.750625
1,Random Forest,0.806122,0.344828,0.740072
2,XGBoost,0.829932,0.431818,0.741924


# Best Model Selection

The final model is selected based on:
- Predictive performance
- Generalization capability
- ROC-AUC score
- F1-score balance

The selected model will be used for:
- Risk scoring
- Explainable AI
- Streamlit deployment

In [20]:
# ============================================
# Save Best Performing Model
# ============================================

joblib.dump(xgb_model, "../models/best_model.pkl")

joblib.dump(scaler, "../models/scaler.pkl")

print("Best model and scaler saved successfully.")

Best model and scaler saved successfully.


# Conclusion

In this notebook, multiple machine learning models were successfully developed and evaluated for employee attrition prediction.

The workflow included:
- Data splitting
- Class balancing using SMOTE
- Feature scaling
- Training multiple classifiers
- Performance evaluation
- Best model selection

Among the evaluated models, the best-performing algorithm demonstrated strong capability in predicting employee attrition risk and is now ready for:
- Explainable AI analysis
- Risk scoring
- Streamlit deployment

Next Phase:
## Explainable AI & SHAP Analysis